# 08. Attention core — MLA and DeepSeek-V4 CSA/HCA

The tensor sizes are small, but the disclosed computation graph is kept:
shared `K=V` MQA, low-rank Q, partial RoPE + conjugate output rotation,
attention sink, grouped output projection, `Ca/Cb` CSA compression,
Lightning Indexer, HCA compression, sliding-window KV, causal masking,
and the V4 layer schedule `sliding → sliding → CSA → HCA → ...`.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def rms_no_weight(x, eps=1e-6):
    scale = x.float().square().mean(-1, keepdim=True).add(eps).rsqrt()
    return (x.float() * scale).to(x.dtype)

def partial_rope(x, positions, rope_dim):
    if rope_dim == 0:
        return x
    content, rope = x[..., :-rope_dim], x[..., -rope_dim:]
    idx = torch.arange(0, rope_dim, 2, device=x.device, dtype=torch.float32)
    inv = 1.0 / (10000 ** (idx / rope_dim))
    angle = positions.float()[:, None] * inv[None]
    cos = angle.cos().repeat_interleave(2, -1)[None, :, None]
    sin = angle.sin().repeat_interleave(2, -1)[None, :, None]
    a, b = rope[..., 0::2], rope[..., 1::2]
    rotate_half = torch.stack([-b, a], -1).flatten(-2)
    rotated = (rope.float() * cos + rotate_half.float() * sin).to(x.dtype)
    return torch.cat([content, rotated], -1)

class GroupedOutput(nn.Module):
    def __init__(self, heads=4, head_dim=8, groups=2, rank=8, model_dim=32):
        super().__init__()
        width = heads * head_dim
        assert width % groups == 0
        self.group_width = width // groups
        self.down = nn.ModuleList(
            [nn.Linear(self.group_width, rank, bias=False) for _ in range(groups)]
        )
        self.up = nn.Linear(groups * rank, model_dim, bias=False)

    def forward(self, heads):
        chunks = heads.flatten(2).split(self.group_width, -1)
        low_rank = [proj(chunk) for proj, chunk in zip(self.down, chunks)]
        return self.up(torch.cat(low_rank, -1))


## 1. MLA baseline


In [ ]:
class TinyMLA(nn.Module):
    def __init__(self, d=32, heads=4, q_rank=12, kv_rank=8, rope_dim=2):
        super().__init__()
        self.heads, self.head_dim, self.rope_dim = heads, 8, rope_dim
        self.q_down = nn.Linear(d, q_rank, bias=False)
        self.q_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(q_rank, heads * self.head_dim, bias=False)
        self.kv_down = nn.Linear(d, kv_rank + rope_dim, bias=False)
        self.kv_norm = nn.RMSNorm(kv_rank)
        self.kv_up = nn.Linear(kv_rank, heads * 12, bias=False)
        self.out = nn.Linear(heads * 6, d, bias=False)

    def forward(self, x):
        b, t, _ = x.shape
        pos = torch.arange(t, device=x.device)
        q = self.q_up(self.q_norm(self.q_down(x))).view(b, t, self.heads, 8)
        q = partial_rope(q, pos, self.rope_dim).transpose(1, 2)
        latent, shared_rope = self.kv_down(x).split([8, self.rope_dim], -1)
        kv = self.kv_up(self.kv_norm(latent)).view(b, t, self.heads, 12)
        k_content, value = kv.split([6, 6], -1)
        rope = shared_rope[:, :, None].expand(-1, -1, self.heads, -1)
        rope = partial_rope(rope, pos, self.rope_dim)
        key = torch.cat([k_content, rope[..., -self.rope_dim:]], -1).transpose(1, 2)
        value = value.transpose(1, 2)
        y = F.scaled_dot_product_attention(q, key, value, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().flatten(2))

x = torch.randn(2, 16, 32, device=device)
print("MLA:", TinyMLA().to(device)(x).shape)


## 2. V4 compressors and Lightning Indexer


In [ ]:
class CSACompressor(nn.Module):
    # Ca from the previous block + Cb from the current block.
    def __init__(self, d=32, head_dim=8, rate=4):
        super().__init__()
        self.rate, self.head_dim = rate, head_dim
        self.kv = nn.Linear(d, 2 * head_dim, bias=False)
        self.gate = nn.Linear(d, 2 * head_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(rate, 2 * head_dim))
        self.norm = nn.RMSNorm(head_dim)

    def forward(self, x):
        b, t, _ = x.shape
        usable = (t // self.rate) * self.rate
        x = x[:, :usable]
        if usable == 0:
            return x.new_zeros(b, 0, self.head_dim), torch.empty(
                0, dtype=torch.long, device=x.device
            )
        n = usable // self.rate
        kv = self.kv(x).view(b, n, self.rate, 2 * self.head_dim)
        gate = self.gate(x).view_as(kv) + self.bias
        ca, cb = kv[..., :self.head_dim], kv[..., self.head_dim:]
        ga, gb = gate[..., :self.head_dim], gate[..., self.head_dim:]
        slots = kv.new_zeros(b, n, 2 * self.rate, self.head_dim)
        logits = gate.new_full(slots.shape, float("-inf"))
        slots[:, :, self.rate:] = cb
        logits[:, :, self.rate:] = gb
        if n > 1:
            slots[:, 1:, :self.rate] = ca[:, :-1]
            logits[:, 1:, :self.rate] = ga[:, :-1]
        weight = logits.softmax(2, dtype=torch.float32).to(slots.dtype)
        compressed = self.norm((slots * weight).sum(2))
        positions = torch.arange(n, device=x.device) * self.rate
        return compressed, positions

class HCACompressor(nn.Module):
    def __init__(self, d=32, head_dim=8, rate=6):
        super().__init__()
        self.rate, self.head_dim = rate, head_dim
        self.kv = nn.Linear(d, head_dim, bias=False)
        self.gate = nn.Linear(d, head_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(rate, head_dim))
        self.norm = nn.RMSNorm(head_dim)

    def forward(self, x):
        b, t, _ = x.shape
        usable = (t // self.rate) * self.rate
        x = x[:, :usable]
        if usable == 0:
            return x.new_zeros(b, 0, self.head_dim), torch.empty(
                0, dtype=torch.long, device=x.device
            )
        n = usable // self.rate
        kv = self.kv(x).view(b, n, self.rate, self.head_dim)
        gate = self.gate(x).view_as(kv) + self.bias
        weight = gate.softmax(2, dtype=torch.float32).to(kv.dtype)
        compressed = self.norm((kv * weight).sum(2))
        positions = torch.arange(n, device=x.device) * self.rate
        return compressed, positions

class LightningIndexer(nn.Module):
    def __init__(self, d=32, q_rank=12, heads=3, head_dim=8, top_k=2):
        super().__init__()
        self.heads, self.head_dim, self.top_k, self.rate = heads, head_dim, top_k, 4
        self.compressor = CSACompressor(d, head_dim, self.rate)
        self.q_up = nn.Linear(q_rank, heads * head_dim, bias=False)
        self.head_weight = nn.Linear(d, heads, bias=False)

    def forward(self, hidden, q_residual, rope_dim=2):
        b, t, _ = hidden.shape
        pos = torch.arange(t, device=hidden.device)
        key, key_pos = self.compressor(hidden)
        key = partial_rope(key[:, :, None], key_pos, rope_dim).squeeze(2)
        q = self.q_up(q_residual).view(b, t, self.heads, self.head_dim)
        q = partial_rope(q, pos, rope_dim)
        per_head = torch.einsum("bthd,bnd->bthn", q.float(), key.float())
        per_head = F.relu(per_head) / math.sqrt(self.head_dim)
        weight = self.head_weight(hidden).float() / math.sqrt(self.heads)
        scores = (per_head * weight[..., None]).sum(2)
        if key.size(1) == 0:
            return torch.empty(b, t, 0, dtype=torch.long, device=hidden.device), scores
        threshold = (pos[None] + 1) // self.rate
        entries = torch.arange(key.size(1), device=hidden.device)
        scores = scores.masked_fill(entries[None, None] >= threshold[..., None], float("-inf"))
        k = min(self.top_k, key.size(1))
        selected = scores.topk(k, -1).indices
        invalid = selected >= threshold[..., None]
        return torch.where(invalid, -torch.ones_like(selected), selected), scores


## 3. Shared-KV MQA attention core


In [ ]:
class V4Attention(nn.Module):
    def __init__(self, kind="sliding", d=32, heads=4, head_dim=8, rope_dim=2):
        super().__init__()
        assert kind in {"sliding", "csa", "hca"}
        self.kind, self.heads, self.head_dim, self.rope_dim = kind, heads, head_dim, rope_dim
        self.window, self.q_rank = 4, 12
        self.q_down = nn.Linear(d, self.q_rank, bias=False)
        self.q_norm = nn.RMSNorm(self.q_rank)
        self.q_up = nn.Linear(self.q_rank, heads * head_dim, bias=False)
        self.kv = nn.Linear(d, head_dim, bias=False)
        self.kv_norm = nn.RMSNorm(head_dim)
        self.sinks = nn.Parameter(torch.zeros(heads))
        self.output = GroupedOutput(heads, head_dim, 2, 8, d)
        self.compressor = CSACompressor(d, head_dim, 4) if kind == "csa" else (
            HCACompressor(d, head_dim, 6) if kind == "hca" else None
        )
        self.indexer = LightningIndexer(d, self.q_rank) if kind == "csa" else None

    def query(self, x, pos):
        latent = self.q_norm(self.q_down(x))
        q = self.q_up(latent).view(x.size(0), x.size(1), self.heads, self.head_dim)
        return partial_rope(rms_no_weight(q), pos, self.rope_dim), latent

    def shared_kv(self, x, pos):
        kv = self.kv_norm(self.kv(x))[:, :, None]
        return partial_rope(kv, pos, self.rope_dim).squeeze(2)

    def attend(self, q, kv, valid):
        score = torch.einsum("bhd,bkd->bhk", q, kv) / math.sqrt(self.head_dim)
        score = score.masked_fill(~valid[:, None], torch.finfo(score.dtype).min)
        sink = self.sinks[None, :, None].expand(q.size(0), -1, 1)
        logits = torch.cat([score, sink], -1)
        logits = logits - logits.max(-1, keepdim=True).values
        weight = logits.softmax(-1)[..., :-1]
        return torch.einsum("bhk,bkd->bhd", weight, kv)

    def forward(self, x):
        b, t, _ = x.shape
        pos = torch.arange(t, device=x.device)
        q, q_latent = self.query(x, pos)
        local = self.shared_kv(x, pos)  # one shared K=V head
        comp, comp_pos, selected, index_scores = None, None, None, None
        if self.compressor is not None:
            comp, comp_pos = self.compressor(x)
            comp = partial_rope(comp[:, :, None], comp_pos, self.rope_dim).squeeze(2)
        if self.kind == "csa":
            selected, index_scores = self.indexer(x, q_latent, self.rope_dim)

        outputs = []
        for token in range(t):
            start = max(0, token - self.window + 1)
            parts = [local[:, start:token + 1]]
            masks = [torch.ones(b, token - start + 1, dtype=torch.bool, device=x.device)]
            if self.kind == "csa" and selected.size(-1) > 0:
                ids = selected[:, token]
                valid = ids >= 0
                safe = ids.clamp_min(0)
                batch_ids = torch.arange(b, device=x.device)[:, None]
                parts.append(comp[batch_ids, safe])
                masks.append(valid)
            elif self.kind == "hca" and comp.size(1) > 0:
                threshold = (token + 1) // self.compressor.rate
                parts.append(comp)
                ids = torch.arange(comp.size(1), device=x.device)
                masks.append((ids[None] < threshold).expand(b, -1))
            kv = torch.cat(parts, 1)
            valid = torch.cat(masks, 1)
            outputs.append(self.attend(q[:, token], kv, valid))

        heads = torch.stack(outputs, 1)
        heads = partial_rope(heads, -pos, self.rope_dim)  # conjugate output rotation
        return self.output(heads), {
            "selected": selected, "index_scores": index_scores, "compressed_pos": comp_pos
        }, q_latent


## 4. Causality, indexer warm-up, and V4 layer schedule


In [ ]:
test = torch.randn(2, 16, 32, device=device)
csa = V4Attention("csa").to(device)
csa_out, diag, _ = csa(test)

violations = 0
for token in range(test.size(1)):
    ids = diag["selected"][:, token]
    valid = ids >= 0
    if valid.any():
        violations += int((ids[valid] >= (token + 1) // 4).sum())
print("CSA future selections:", violations)
assert violations == 0

# Dense warm-up target for the non-differentiable Lightning top-k selector.
pos = torch.arange(test.size(1), device=device)
core_comp, core_pos = csa.compressor(test)
core_comp = partial_rope(core_comp[:, :, None], core_pos, csa.rope_dim).squeeze(2)
core_q, _ = csa.query(test, pos)
teacher = torch.einsum("bthd,bnd->bthn", core_q.detach(), core_comp.detach()).sum(2)
losses = []
for token in range(test.size(1)):
    n = (token + 1) // 4
    if n:
        target = teacher[:, token, :n].softmax(-1)
        student = diag["index_scores"][:, token, :n].log_softmax(-1)
        losses.append(F.kl_div(student, target, reduction="batchmean"))
warmup_loss = torch.stack(losses).mean()
csa.zero_grad()
warmup_loss.backward()
print("indexer warm-up:", warmup_loss.item())
print("indexer grad:", csa.indexer.q_up.weight.grad.norm().item())

hca = V4Attention("hca").to(device)
prefix_end = 7
perturbed = test.clone()
perturbed[:, prefix_end + 1:] = torch.randn_like(perturbed[:, prefix_end + 1:])
with torch.no_grad():
    original = hca(test)[0][:, :prefix_end + 1]
    changed = hca(perturbed)[0][:, :prefix_end + 1]
future_effect = (original - changed).abs().max().item()
print("HCA prefix change from future perturbation:", future_effect)
assert future_effect < 1e-6

class TinyV4AttentionStack(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.pattern = ["sliding", "sliding", "csa", "hca", "csa", "hca"]
        self.norms = nn.ModuleList([nn.RMSNorm(d) for _ in self.pattern])
        self.layers = nn.ModuleList([V4Attention(kind, d=d) for kind in self.pattern])

    def forward(self, x):
        for norm, layer in zip(self.norms, self.layers):
            x = x + layer(norm(x))[0]
        return x

stack = TinyV4AttentionStack().to(device)
out = stack(test)
loss = out.square().mean()
loss.backward()
print("V4 pattern:", stack.pattern)
print("stack output:", out.shape)
print("CSA sink grad:", stack.layers[2].sinks.grad.norm().item())
print("HCA output-proj grad:", stack.layers[3].output.down[0].weight.grad.norm().item())


## References and provenance

DeepSeek-V4's disclosed attention differs materially from ordinary attention:
the KV side is a single shared `K=V` head; CSA uses `Ca/Cb` overlap compression
and a separate Lightning Indexer with `Σ_h w_h ReLU(q_h·k)` scoring; HCA uses
heavier gated compression; both retain a local sliding branch; partial RoPE is
undone with the conjugate rotation on the output; an attention sink and grouped
output projection are used. V4 begins with sliding-attention layers and then
interleaves CSA and HCA. A short Lightning Indexer warm-up precedes sparse training.

Only execution scale is reduced here; the above paths are not replaced by generic attention.
